# 🛡️ "Is That Really Them?" — Deepfake Scam Ad Detector
### Optimized for Google Colab NVIDIA T4 GPU (2-Hour Budget)

This notebook trains an **EfficientNet-B0** deepfake detector on uniformly sampled face crops, performs **Top-3 Mean** video-level aggregation, generates **Grad-CAM attention heatmaps**, and evaluates ROC-AUC / F1 metrics.

## 1. Verify NVIDIA T4 GPU & Environment

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Clone / Setup Deepfake Detector Repository & Dependencies

In [ ]:
# If running directly inside the cloned repository, install dependencies:
!pip install --upgrade pip
!pip install torch torchvision opencv-python-headless pandas scikit-learn matplotlib seaborn pyyaml datasets huggingface_hub

## 3. Run Pipeline Tests (Smoke Test)

In [ ]:
!python -m pytest -v tests/test_pipeline.py

## 4. Run End-to-End Synthetic Benchmark (Fast 1-Minute Verification)

In [ ]:
!python src/train.py --synthetic --epochs 2 --batch_size 4

## 5. Train Full Model on Hugging Face Streaming Dataset (`belkhir-nacim/deepfake-videos`)
Uses Phase 1 (Frozen Backbone Warmup) & Phase 2 (Differential Learning Rate Fine-tuning) with AMP on T4.

In [ ]:
!python src/train.py --config configs/config.yaml

## 6. Visualize Results (Loss Curves, ROC, Precision-Recall & Confusion Matrix)

In [ ]:
import os
from IPython.display import Image, display

output_plots = [
    "outputs/training_curves.png",
    "outputs/confusion_matrix.png",
    "outputs/roc_curve.png",
    "outputs/pr_curve.png"
]

for plot_path in output_plots:
    if os.path.exists(plot_path):
        print(f"\n--- {plot_path} ---")
        display(Image(filename=plot_path))

## 7. Display Grad-CAM Explainability Triptych
Visualizes **Original Face Frame | Grad-CAM Heatmap | Attention Overlay** on suspicious frames.

In [ ]:
import glob
cam_images = glob.glob("outputs/explanations/*.png") + glob.glob("outputs/*.png")
cam_images = [img for img in cam_images if "triptych" in img or "gradcam" in img]

if cam_images:
    for cam_img in cam_images[:3]:
        print(f"\nGrad-CAM Triptych: {cam_img}")
        display(Image(filename=cam_img))
else:
    print("No Grad-CAM images found in outputs/ yet. Run step 4 or 5 first!")